# 07 · Financial Impact Simulation

Digital Onboarding Optimization Case Study

> Fully synthetic dataset. No real company, customer, or proprietary information is included.

## 1. Objective

This notebook translates funnel improvement into business impact.

The financial model is intentionally simple and portfolio-friendly. It estimates the value of:

- More approved users
- Fewer support contacts
- Lower operational handling cost
- Incremental contribution margin
- ROI and payback

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

users = pd.read_csv(DATA_DIR / "onboarding_users.csv", parse_dates=["signup_timestamp"])
events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=["event_timestamp"])
support = pd.read_csv(DATA_DIR / "support_calls.csv", parse_dates=["call_timestamp"])

users["has_support_call"] = users["user_id"].isin(support["user_id"]).astype(int)
users["upload_now"] = users["upload_choice"].eq("upload_now").astype(int)

users.head()

## 2. Business Assumptions

In [ ]:
assumptions = {
    "monthly_new_users": 100_000,
    "avg_monthly_revenue_per_approved_user": 18.50,
    "gross_margin": 0.62,
    "avg_support_cost_per_call": 6.20,
    "implementation_cost": 45_000,
    "monthly_maintenance_cost": 5_000,
    "analysis_horizon_months": 12,
    "expected_relative_lift_upload_now": 0.12
}

pd.Series(assumptions).to_frame("value")

## 3. Baseline Funnel Economics

In [ ]:
baseline_upload_now = users["upload_now"].mean()
baseline_approval = users["approved"].mean()
baseline_support = users["has_support_call"].mean()

baseline_monthly_approved = assumptions["monthly_new_users"] * baseline_approval
baseline_monthly_support_calls = assumptions["monthly_new_users"] * baseline_support
baseline_monthly_margin = baseline_monthly_approved * assumptions["avg_monthly_revenue_per_approved_user"] * assumptions["gross_margin"]
baseline_monthly_support_cost = baseline_monthly_support_calls * assumptions["avg_support_cost_per_call"]

baseline_economics = pd.Series({
    "baseline_upload_now_rate": baseline_upload_now,
    "baseline_approval_rate": baseline_approval,
    "baseline_support_rate": baseline_support,
    "baseline_monthly_approved_users": baseline_monthly_approved,
    "baseline_monthly_support_calls": baseline_monthly_support_calls,
    "baseline_monthly_margin": baseline_monthly_margin,
    "baseline_monthly_support_cost": baseline_monthly_support_cost
}).to_frame("baseline")

baseline_economics

## 4. Scenario Simulation

In [ ]:
approval_now = users.loc[users["upload_now"] == 1, "approved"].mean()
approval_later = users.loc[users["upload_now"] == 0, "approved"].mean()
support_now = users.loc[users["upload_now"] == 1, "has_support_call"].mean()
support_later = users.loc[users["upload_now"] == 0, "has_support_call"].mean()

def simulate_scenario(relative_lift_upload_now):
    new_upload_now = min(baseline_upload_now * (1 + relative_lift_upload_now), 0.95)
    new_upload_later = 1 - new_upload_now

    new_approval_rate = new_upload_now * approval_now + new_upload_later * approval_later
    new_support_rate = new_upload_now * support_now + new_upload_later * support_later

    monthly_approved = assumptions["monthly_new_users"] * new_approval_rate
    monthly_support_calls = assumptions["monthly_new_users"] * new_support_rate

    monthly_margin = monthly_approved * assumptions["avg_monthly_revenue_per_approved_user"] * assumptions["gross_margin"]
    monthly_support_cost = monthly_support_calls * assumptions["avg_support_cost_per_call"]

    incremental_margin = monthly_margin - baseline_monthly_margin
    support_savings = baseline_monthly_support_cost - monthly_support_cost
    monthly_net_benefit = incremental_margin + support_savings - assumptions["monthly_maintenance_cost"]

    annual_net_benefit = monthly_net_benefit * assumptions["analysis_horizon_months"] - assumptions["implementation_cost"]
    roi = annual_net_benefit / assumptions["implementation_cost"]
    payback_months = assumptions["implementation_cost"] / max(monthly_net_benefit, 1)

    return {
        "relative_lift_upload_now": relative_lift_upload_now,
        "new_upload_now_rate": new_upload_now,
        "new_approval_rate": new_approval_rate,
        "new_support_rate": new_support_rate,
        "incremental_approved_users_month": monthly_approved - baseline_monthly_approved,
        "support_calls_avoided_month": baseline_monthly_support_calls - monthly_support_calls,
        "incremental_margin_month": incremental_margin,
        "support_savings_month": support_savings,
        "monthly_net_benefit": monthly_net_benefit,
        "annual_net_benefit": annual_net_benefit,
        "roi": roi,
        "payback_months": payback_months
    }

scenarios = pd.DataFrame([
    simulate_scenario(0.05),
    simulate_scenario(0.10),
    simulate_scenario(0.12),
    simulate_scenario(0.15),
    simulate_scenario(0.20)
])

scenarios

## 5. Impact Visualization

In [ ]:
ax = scenarios.set_index("relative_lift_upload_now")["annual_net_benefit"].plot(kind="bar", figsize=(10, 5))
ax.set_title("Annual Net Benefit by Upload-now Lift Scenario")
ax.set_ylabel("Annual net benefit")
ax.set_xlabel("Relative upload-now lift")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
ax = scenarios.set_index("relative_lift_upload_now")[["incremental_approved_users_month", "support_calls_avoided_month"]].plot(kind="bar", figsize=(10, 5))
ax.set_title("Monthly Business Impact by Scenario")
ax.set_ylabel("Users / Calls")
ax.set_xlabel("Relative upload-now lift")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Sensitivity Analysis

In [ ]:
lifts = np.linspace(0.02, 0.25, 12)
revenues = [12, 18.5, 25, 35]

sensitivity = []
for lift in lifts:
    for revenue in revenues:
        original_revenue = assumptions["avg_monthly_revenue_per_approved_user"]
        assumptions["avg_monthly_revenue_per_approved_user"] = revenue
        out = simulate_scenario(lift)
        out["avg_monthly_revenue_per_approved_user"] = revenue
        sensitivity.append(out)
        assumptions["avg_monthly_revenue_per_approved_user"] = original_revenue

sensitivity = pd.DataFrame(sensitivity)
sensitivity.head()

In [ ]:
pivot = sensitivity.pivot_table(
    index="relative_lift_upload_now",
    columns="avg_monthly_revenue_per_approved_user",
    values="annual_net_benefit"
)

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(pivot)
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.columns)
ax.set_yticklabels([f"{x:.0%}" for x in pivot.index])
ax.set_title("Sensitivity Analysis - Annual Net Benefit")
ax.set_xlabel("Avg monthly revenue per approved user")
ax.set_ylabel("Upload-now relative lift")
fig.colorbar(im)
plt.tight_layout()
plt.show()

pivot

## 7. Executive Conclusion

The financial model shows that even modest improvements in immediate document submission can generate measurable value.

The strongest value levers are:

1. Higher approval conversion
2. Reduced support contact demand
3. Lower onboarding friction
4. Higher activation quality

This supports prioritizing the UX intervention and measuring it through a controlled experiment.